# 03 - 误差分析（基于预测结果）

读取推理生成的 `predictions.csv`，统计 TP/FP/FN/TN，并导出漏报（FN）、误报（FP）清单。

**路径解析**：与 CLI 一致，优先 `runs/<run_name>/.latest_infer_session`，否则根目录 `predictions.csv`。

**自动补齐链路**：若尚无 `predictions.csv`，本 Notebook 会按 **`exp_baseline.yaml`** 自动补齐 preprocess（含 validate）→ train → infer（适合快速分析，**全量/多卡请用 `04_run_pipelines.ipynb`**）。补齐前会调用 **`check_preprocess_raw_inputs`**；若 splits 与 `model.yaml` 的 C/H/W 不一致会重跑 preprocess。

**Notebook 提示**：改源码后请 **Kernel → Restart**。若找不到 checkpoint，常见原因是验证集无正例导致 avg_precision 为 nan；训练器仍会在末轮兜底保存权重。

**输出**：`outputs/tables/error_confusion_counts.csv`、`error_FN_samples.csv`、`error_FP_samples.csv`

**流水线关系**：需先 **infer** 产出 `predictions.csv`；完整冒烟/全量流程见 **`04_run_pipelines.ipynb`**（本 Notebook 专注预测误差分析）。


In [ ]:
from pathlib import Path
import sys
# 自动判断项目根目录
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
_SRC = str(PROJECT_ROOT / "src")
# 把 src/ 目录加入 Python 模块搜索路径（优先从 src 找模块）
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

# Jupyter 会把已 import 的模块缓存在 sys.modules；清除后下次 import 会重新加载仓库 src 下的源码
for _name in list(sys.modules):
    if _name == "dram2cnn" or _name.startswith("dram2cnn."):
        del sys.modules[_name]

# 03 默认分析 exp_baseline 的 predictions；全量跑完后可改为 exp_full_best_ddp.yaml
EXP_YAML = PROJECT_ROOT / "configs" / "experiments" / "exp_baseline.yaml"
# 创建输出表格目录（不存在就自动创建）
OUT_T = PROJECT_ROOT / "outputs" / "tables"
OUT_T.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import subprocess
from pathlib import Path

import numpy as np

from dram2cnn.utils.config import load_experiment_config
from dram2cnn.utils.run_layout import read_pointer_dir, resolve_checkpoint_dir

try:
    _EXP_REL_STR = str(EXP_YAML.relative_to(PROJECT_ROOT))
except ValueError:
    _EXP_REL_STR = str(EXP_YAML.resolve())


def _run_shell_script(rel_script: str, arg: str) -> None:
    cmd = ["bash", str(PROJECT_ROOT / rel_script), arg]
    env = os.environ.copy()
    src_abs = str(PROJECT_ROOT / "src")
    pp = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = f"{src_abs}:{pp}" if pp else src_abs
    print("提示：执行", " ".join(cmd))
    subprocess.run(cmd, cwd=str(PROJECT_ROOT), env=env, check=True)


exp_path = EXP_YAML.resolve()
from dram2cnn.utils.preprocess_preflight import experiment_run_name

run_name = experiment_run_name(exp_path)
run_root = PROJECT_ROOT / "runs" / run_name
run_root.mkdir(parents=True, exist_ok=True)

infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
pred_path = None
if infer_dir is not None:
    cand = infer_dir / "predictions.csv"
    if cand.is_file():
        pred_path = cand
if pred_path is None and (run_root / "predictions.csv").is_file():
    infer_dir = run_root
    pred_path = infer_dir / "predictions.csv"

cfg = None
if pred_path is not None and pred_path.is_file():
    print("提示：已找到 predictions.csv，跳过预处理、训练与推理。")
    try:
        cfg = load_experiment_config(str(exp_path))
        from dram2cnn.utils.experiment_guards import assert_predictions_match_experiment

        assert_predictions_match_experiment(cfg, infer_dir, PROJECT_ROOT)
        print("提示：predictions_manifest 与当前 EXP_YAML 一致。")
    except FileNotFoundError as e:
        print(f"提示：无法加载实验配置做 manifest 校验（{e}）；若 predictions 来自其它实验，请改 EXP_YAML 或重跑 infer。")
    except ValueError as e:
        raise RuntimeError(f"predictions 与 EXP_YAML 不一致: {e}") from e
else:
    try:
        cfg = load_experiment_config(str(exp_path))
    except FileNotFoundError as e:
        raise FileNotFoundError(
            "加载实验配置失败（常见原因：data.yaml 中的 mcelog/tickets/index_features 等路径不存在）。\n"
            f"原始错误: {e}\n"
            "若仅需误差分析，请先将 predictions.csv 放到 runs/<run_name>/.latest_infer_session/ 下。"
        ) from e
    run_name = str(cfg["base"].get("run_name", run_name))
    run_root = PROJECT_ROOT / "runs" / run_name
    run_root.mkdir(parents=True, exist_ok=True)

    splits_dir = Path(cfg["data"]["splits_dir"])
    if not splits_dir.is_absolute():
        splits_dir = PROJECT_ROOT / splits_dir

    def _split_x_shape_matches_model(sd: Path, model_cfg: dict) -> bool:
        xt = sd / "X_train.npy"
        if not xt.is_file():
            return False
        x = np.load(xt, mmap_mode="r")
        if x.ndim != 4:
            return False
        exp_c = int(model_cfg.get("in_channels", x.shape[1]))
        exp_h = int(model_cfg.get("grid_height", x.shape[2]))
        exp_w = int(model_cfg.get("grid_width", x.shape[3]))
        _n, c, h, w = x.shape
        return int(c) == exp_c and int(h) == exp_h and int(w) == exp_w

    from dram2cnn.utils.experiment_guards import splits_match_processed

    def _splits_ready(sd: Path) -> bool:
        return splits_match_processed(cfg, PROJECT_ROOT)

    if not _splits_ready(splits_dir):
        from dram2cnn.utils.preprocess_preflight import check_preprocess_raw_inputs

        check_preprocess_raw_inputs(exp_path)
        print("提示：data/splits 不完整，自动运行预处理（耗时取决于原始数据规模）。")
        _run_shell_script("scripts/preprocess.sh", _EXP_REL_STR)
    elif not _split_x_shape_matches_model(splits_dir, cfg["model"]):
        x = np.load(splits_dir / "X_train.npy", mmap_mode="r")
        chw = tuple(int(x.shape[i]) for i in range(1, 4)) if x.ndim == 4 else x.shape
        mc = cfg["model"]
        exp = (int(mc.get("in_channels", 3)), int(mc.get("grid_height", 0)), int(mc.get("grid_width", 0)))
        print(
            f"提示：现有切分 X_train (C,H,W)={chw} 与 model.yaml 期望 {exp} 不一致，将重新运行 preprocess。"
        )
        _run_shell_script("scripts/preprocess.sh", _EXP_REL_STR)

    if not _splits_ready(splits_dir):
        raise RuntimeError(f"预处理后仍缺少 splits，请检查 data/raw 与配置：{splits_dir}")
    if not _split_x_shape_matches_model(splits_dir, cfg["model"]):
        raise RuntimeError(
            f"预处理后 X_train 形状仍与 model.yaml 不一致：{splits_dir}。"
            "请检查 grid_representation / model.in_channels|grid_height|grid_width 并重跑 preprocess。"
        )

    try:
        resolve_checkpoint_dir(PROJECT_ROOT, cfg["base"])
    except FileNotFoundError:
        print("提示：未找到 checkpoint，自动运行训练（可能非常久；与 scripts/train.sh 一致，含 DDP 逻辑）。")
        _run_shell_script("scripts/train.sh", _EXP_REL_STR)

    try:
        resolve_checkpoint_dir(PROJECT_ROOT, cfg["base"])
    except FileNotFoundError as exc:
        raise RuntimeError(f"训练结束后仍未解析到 checkpoint：{exc}") from exc

    infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
    pred_path = None
    if infer_dir is not None:
        cand = infer_dir / "predictions.csv"
        if cand.is_file():
            pred_path = cand
    if pred_path is None and (run_root / "predictions.csv").is_file():
        infer_dir = run_root
        pred_path = infer_dir / "predictions.csv"

    if pred_path is None or not pred_path.is_file():
        print("提示：未找到 predictions.csv，自动运行推理。")
        _run_shell_script("scripts/infer.sh", _EXP_REL_STR)
        infer_dir = read_pointer_dir(run_root, ".latest_infer_session", "latest_infer")
        pred_path = None
        if infer_dir is not None:
            cand = infer_dir / "predictions.csv"
            if cand.is_file():
                pred_path = cand
        if pred_path is None and (run_root / "predictions.csv").is_file():
            infer_dir = run_root
            pred_path = infer_dir / "predictions.csv"

    if pred_path is None or not pred_path.is_file():
        raise RuntimeError(f"推理后仍找不到 predictions.csv，请检查 {run_root}")

infer_dir = pred_path.parent
print("run_name:", run_name)
print("infer_dir:", infer_dir)
print("predictions:", pred_path)


In [ ]:
import pandas as pd

if "pred_path" not in globals() or pred_path is None:
    raise RuntimeError("请先自上而下运行上一单元（须成功解析 predictions）。")

df = pd.read_csv(pred_path)
need = {"y_true", "y_pred"}
if not need.issubset(df.columns):
    raise ValueError(f"predictions.csv 需含列 {need}，实际: {list(df.columns)}")

yt = df["y_true"].astype(int)
yp = df["y_pred"].astype(int)

tp = int(((yt == 1) & (yp == 1)).sum())
fp = int(((yt == 0) & (yp == 1)).sum())
fn = int(((yt == 1) & (yp == 0)).sum())
tn = int(((yt == 0) & (yp == 0)).sum())

summary = pd.DataFrame([
    {"name": "TP", "count": tp},
    {"name": "FP", "count": fp},
    {"name": "FN", "count": fn},
    {"name": "TN", "count": tn},
])
summary.to_csv(OUT_T / "error_confusion_counts.csv", index=False)
print(summary)

fn_df = df[(yt == 1) & (yp == 0)].copy()
fp_df = df[(yt == 0) & (yp == 1)].copy()
fn_df.to_csv(OUT_T / "error_FN_samples.csv", index=False)
fp_df.to_csv(OUT_T / "error_FP_samples.csv", index=False)
print("FN rows", len(fn_df), "FP rows", len(fp_df))
print("已写入 outputs/tables/error_*.csv")
